###Preparación del entorno
Esta celda descarga el proyecto desde GitHub e instala las dependencias que necesita para funcionar en Colab. Al final, ejecuta una prueba de búsqueda para comprobar que todo quedó listo antes de continuar con el desarrollo.


In [ ]:
%%bash
set -euo pipefail

project_dir=/content/softtek-knowledge-hub

if [ ! -d "$project_dir/.git" ]; then
  git clone --quiet --branch main \
    https://github.com/ierisaac2005-png/softtek-knowledge-hub.git "$project_dir"
fi

cd "$project_dir"
python -m venv --without-pip .venv
python -m pip --python .venv/bin/python install \
  --quiet --disable-pip-version-check -r requirements.txt

printf 'Rama: %s\n' "$(git branch --show-current)"
.venv/bin/python --version
.venv/bin/python exercises/ex5_retrieval_evaluation.py

Rama: fase-2
Python 3.13.18
Query                                               RR base   RR new
how do I deploy a service to kubernetes                1.00     1.00
what happened when the payment system went down        0.00     1.00
rules for working from home                            1.00     1.00
course to learn security and avoid phishing            0.33     1.00
is the project over budget                             1.00     1.00
login tokens and identity provider                     1.00     1.00
the website broke after a release                      1.00     1.00
who approves changes to production                     1.00     1.00

Baseline TF-IDF                            P@3 = 0.375   MRR = 0.792
TF-IDF + stemming + ontology expansion     P@3 = 0.417   MRR = 1.000


## Prueba del clasificador actual

Antes de modificar el clasificador, probamos tres textos en inglés. Incluimos un incidente claro, un documento de capacitación con vocabulario de proyectos y un texto fuera del tema. Para cada caso registramos la categoría esperada, la predicción del modelo y el puntaje que le asignó.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

MPLBACKEND=Agg PYTHONPATH=src .venv/bin/python - <<'PY'
from classifier import MODELS, build_pipeline, classify, load_documents

documents = load_documents()
model = build_pipeline(MODELS["Logistic Regression"]()).fit(
    documents["content"], documents["category"]
)

cases = [
    (
        "Incidente claro",
        "The checkout service stopped accepting payments after the release. "
        "The on-call team rolled back the change and restored service.",
        "Incident Reports",
    ),
    (
        "Capacitación con vocabulario de proyectos",
        "This onboarding lesson shows new team members how to deploy a service "
        "to Kubernetes for their first client project.",
        "Training Manuals",
    ),
    (
        "Texto fuera del tema",
        "My dog needs new food this week.",
        "Sin categoría aplicable",
    ),
]

predictions = classify(model, [text for _, text, _ in cases])

for (case, text, expected), (category, score) in zip(cases, predictions):
    print(f"\n{case}")
    print(f"Texto: {text}")
    print(f"Esperado: {expected}")
    print(f"Predicción: {category}")
    print(f"Puntaje del modelo: {score:.1%}")
PY


Incidente claro
Texto: The checkout service stopped accepting payments after the release. The on-call team rolled back the change and restored service.
Esperado: Incident Reports
Predicción: Incident Reports
Puntaje del modelo: 36.3%

Capacitación con vocabulario de proyectos
Texto: This onboarding lesson shows new team members how to deploy a service to Kubernetes for their first client project.
Esperado: Training Manuals
Predicción: Project Reports
Puntaje del modelo: 31.2%

Texto fuera del tema
Texto: My dog needs new food this week.
Esperado: Sin categoría aplicable
Predicción: Training Manuals
Puntaje del modelo: 24.1%


### Resultados de la prueba

- **Incidente claro.** El modelo acertó con `Incident Reports` y le asignó un puntaje de 36.3%.
- **Capacitación con vocabulario de proyectos.** El modelo predijo `Project Reports` con 31.2%, aunque el propósito del texto era enseñar y la categoría esperada era `Training Manuals`.
- **Texto fuera del tema.** El modelo asignó `Training Manuals` con 24.1%, aunque ninguna categoría aplicaba.

Los resultados justifican evaluar una opción de **«Requiere revisión»** para evitar que todos los textos reciban automáticamente una etiqueta definitiva. El incidente correcto también obtuvo un puntaje bajo, así que estos tres casos sirven para detectar el problema. Para elegir un umbral necesitamos más ejemplos con categorías esperadas.

## Evaluación de umbrales de revisión

Probamos distintos puntajes mínimos para aceptar una clasificación automática. Usamos validación cruzada de cinco particiones para que cada documento se evalúe con un modelo que no lo usó para entrenar. Para cada umbral contamos los aciertos y errores aceptados, los aciertos enviados a revisión y los textos fuera del tema que todavía reciben una etiqueta automática.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

MPLBACKEND=Agg PYTHONPATH=src .venv/bin/python - <<'PY'
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from classifier import MODELS, SEED, build_pipeline, load_documents

documents = load_documents()
texts = documents["content"]
expected = documents["category"].to_numpy()

model = build_pipeline(MODELS["Logistic Regression"]())
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
probabilities = cross_val_predict(
    model, texts, expected, cv=folds, method="predict_proba"
)

outside_texts = [
    "My dog needs new food this week.",
    "Bake the cake for thirty minutes and let it cool.",
    "I compared prices for a weekend trip to the beach.",
    "Our school team delivered a history project and presented the final report.",
]

model.fit(texts, expected)
scores = probabilities.max(axis=1)
correct = model.classes_[probabilities.argmax(axis=1)] == expected
outside_scores = model.predict_proba(outside_texts).max(axis=1)

print(f"Documentos evaluados: {len(expected)}")
print(f"Aciertos sin umbral: {correct.sum()}/{len(expected)}")

print("\nPuntajes de los textos fuera del tema")
for text, score in zip(outside_texts, outside_scores):
    print(f"  {score:5.1%} | {text}")

print("\nUmbral | Aciertos aceptados | Errores aceptados | Aciertos en revisión | Fuera del tema aceptados")
for threshold in (0, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50):
    accepted = scores >= threshold
    correct_accepted = (accepted & correct).sum()
    wrong_accepted = (accepted & ~correct).sum()
    correct_reviewed = (~accepted & correct).sum()
    outside_accepted = (outside_scores >= threshold).sum()
    print(
        f"{threshold:6.0%} | {correct_accepted:18} | {wrong_accepted:17} | "
        f"{correct_reviewed:19} | {outside_accepted:24}/{len(outside_texts)}"
    )
PY

Documentos evaluados: 60
Aciertos sin umbral: 54/60

Puntajes de los textos fuera del tema
  24.1% | My dog needs new food this week.
  31.2% | Bake the cake for thirty minutes and let it cool.
  23.0% | I compared prices for a weekend trip to the beach.
  57.7% | Our school team delivered a history project and presented the final report.

Umbral | Aciertos aceptados | Errores aceptados | Aciertos en revisión | Fuera del tema aceptados
    0% |                 54 |                 6 |                   0 |                        4/4
   25% |                 52 |                 6 |                   2 |                        2/4
   30% |                 47 |                 5 |                   7 |                        2/4
   35% |                 42 |                 4 |                  12 |                        1/4
   40% |                 28 |                 0 |                  26 |                        1/4
   45% |                 18 |                 0 |                

### Interpretación de los resultados

Sin umbral, el clasificador acertó en 54 de 60 documentos (90%). Con un umbral de 40%, aceptó automáticamente 28 documentos y acertó en los 28. Los otros 32 quedaron para revisión, incluidos los seis errores del modelo y 26 predicciones que eran correctas. La cobertura automática fue de 46.7%.

Elegimos el 40% como umbral provisional porque fue el menor de los valores probados que envió a revisión los seis errores del corpus. Subirlo a 45% redujo los aciertos aceptados de 28 a 18 y siguió dejando pasar el mismo texto fuera del tema.

El reporte de historia escolar obtuvo 57.7% y superó el umbral pese a quedar fuera del dominio. Por eso usamos el puntaje para decidir cuándo revisar una etiqueta. Para evaluar si un texto pertenece al dominio harán falta ejemplos adicionales.

## Comparación de dos versiones del clasificador

Evaluamos si agregar fragmentos de caracteres al TF-IDF de palabras ayuda a clasificar mejor los documentos. Conservamos la regresión logística, los 60 documentos y las mismas cinco particiones. Comparamos los aciertos totales, los errores por documento y los aciertos cuando ambos modelos aceptan automáticamente la misma cantidad de documentos.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

MPLBACKEND=Agg PYTHONPATH=src .venv/bin/python - <<'PY'
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import FeatureUnion
from classifier import MODELS, SEED, build_pipeline, load_documents

docs = load_documents()
expected = docs["category"].to_numpy()
classes = np.unique(expected)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

original = build_pipeline(MODELS["Logistic Regression"]())
combined = build_pipeline(MODELS["Logistic Regression"]())
combined.set_params(tfidf=FeatureUnion([
    ("words", combined["tfidf"]),
    ("chars", TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True
    )),
]))

predictions = {}
for name, model in [("Original", original), ("Combinado", combined)]:
    probs = cross_val_predict(
        model, docs["content"], expected, cv=folds, method="predict_proba"
    )
    predicted = classes[probs.argmax(axis=1)]
    predictions[name] = predicted
    correct = predicted == expected
    ranked = np.argsort(-probs.max(axis=1), kind="stable")

    print(f"\n{name}")
    print(f"  Aciertos totales: {correct.sum()}/{len(docs)}")
    print(f"  Aciertos al aceptar 46 documentos: {correct[ranked[:46]].sum()}/46")
    print("  Documentos con error:", ", ".join(docs.loc[~correct, "id"]))

print("\nDocumentos donde cambió la predicción")
changed = docs.loc[
    predictions["Original"] != predictions["Combinado"], ["id", "title", "category"]
].copy()
changed["Original"] = predictions["Original"][changed.index]
changed["Combinado"] = predictions["Combinado"][changed.index]
print(changed.to_string(index=False))
PY


Original
  Aciertos totales: 54/60
  Aciertos al aceptar 46 documentos: 42/46
  Documentos con error: TD03, TD07, TD08, PP07, TM03, IR10

Combinado
  Aciertos totales: 57/60
  Aciertos al aceptar 46 documentos: 44/46
  Documentos con error: TD07, TM03, IR10

Documentos donde cambió la predicción
  id                              title                category         Original               Combinado
TD03        Kubernetes Deployment Guide Technical Documentation Training Manuals Technical Documentation
TD08 Mobile App Technical Specification Technical Documentation  Project Reports Technical Documentation
PP07        Change Management Procedure   Policies & Procedures  Project Reports   Policies & Procedures


### Resultado de la comparación

En validación cruzada, el modelo combinado acertó en 57 de 60 documentos (95%), frente a 54 de 60 (90%) del original. Corrigió la clasificación de dos documentos técnicos (TD03 y TD08) y un procedimiento de gestión de cambios (PP07). Ambos modelos siguieron fallando en TD07, TM03 e IR10.

Al aceptar los 46 documentos con mayor puntaje, el modelo combinado logró 44 aciertos frente a 42 del original. Elegimos esta variante para la siguiente prueba: evaluar cómo responde ante textos externos al corpus antes de fijar un umbral de revisión.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

MPLBACKEND=Agg PYTHONPATH=src .venv/bin/python - <<'PY'
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from classifier import MODELS, build_pipeline, classify, load_documents

docs = load_documents()
original = build_pipeline(MODELS["Logistic Regression"]())
combined = build_pipeline(MODELS["Logistic Regression"]())
combined.set_params(tfidf=FeatureUnion([
    ("words", combined["tfidf"]),
    ("chars", TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True
    )),
]))

cases = [
    ("Incidente", "Incident Reports",
     "The checkout service stopped accepting payments after the release. "
     "The on-call team rolled back the change and restored service."),
    ("Capacitación", "Training Manuals",
     "This onboarding lesson shows new team members how to deploy a service "
     "to Kubernetes for their first client project."),
    ("Mascota", "Fuera del dominio", "My dog needs new food this week."),
    ("Receta", "Fuera del dominio", "Bake the cake for thirty minutes and let it cool."),
    ("Viaje", "Fuera del dominio", "I compared prices for a weekend trip to the beach."),
    ("Reporte escolar", "Fuera del dominio",
     "Our school team delivered a history project and presented the final report."),
]

texts = [text for _, _, text in cases]
results = {}
for name, model in [("Original", original), ("Combinado", combined)]:
    model.fit(docs["content"], docs["category"])
    results[name] = classify(model, texts)

for i, (case, expected, _) in enumerate(cases):
    print(f"\n{case} | Esperado: {expected}")
    for name, predictions in results.items():
        category, score = predictions[i]
        status = "Aceptada" if score >= 0.40 else "Requiere revisión"
        print(f"  {name:10} {category:25} {score:6.1%}   {status}")
PY


Incidente | Esperado: Incident Reports
  Original   Incident Reports           36.3%   Requiere revisión
  Combinado  Incident Reports           45.0%   Aceptada

Capacitación | Esperado: Training Manuals
  Original   Project Reports            31.2%   Requiere revisión
  Combinado  Training Manuals           31.9%   Requiere revisión

Mascota | Esperado: Fuera del dominio
  Original   Training Manuals           24.1%   Requiere revisión
  Combinado  Training Manuals           26.4%   Requiere revisión

Receta | Esperado: Fuera del dominio
  Original   Incident Reports           31.2%   Requiere revisión
  Combinado  Incident Reports           32.4%   Requiere revisión

Viaje | Esperado: Fuera del dominio
  Original   Policies & Procedures      23.0%   Requiere revisión
  Combinado  Training Manuals           25.6%   Requiere revisión

Reporte escolar | Esperado: Fuera del dominio
  Original   Project Reports            57.7%   Aceptada
  Combinado  Project Reports            72.1%   

### Prueba con textos externos al corpus

Con el umbral provisional de 40%, el modelo combinado aceptó correctamente el incidente (45.0%) y corrigió la categoría de la capacitación a `Training Manuals`, aunque la envió a revisión (31.9%). Ambos modelos enviaron a revisión los textos sobre la mascota, la receta y el viaje.

La principal limitación apareció en el reporte escolar. Ambos modelos lo aceptaron como `Project Reports`, y el puntaje aumentó de 57.7% a 72.1% con el combinado. En estos ejemplos, un puntaje alto indica que el texto encaja con una categoría aprendida, pero por sí solo no permite decidir si pertenece al ámbito del proyecto.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

MPLBACKEND=Agg PYTHONPATH=src .venv/bin/python - <<'PY'
from sklearn.metrics.pairwise import cosine_similarity
from classifier import MODELS, build_pipeline, load_documents

docs = load_documents()
vectorizer = build_pipeline(MODELS["Logistic Regression"]())["tfidf"]
corpus = vectorizer.fit_transform(docs["content"])

cases = {
    "Reporte escolar":
        "Our school team delivered a history project and presented the final report.",
    "Reporte de proyecto de TI":
        "The integration project is two weeks behind schedule. The client approved "
        "a revised budget and a new delivery milestone.",
    "Capacitación":
        "This onboarding lesson shows new team members how to deploy a service "
        "to Kubernetes for their first client project.",
}

for name, text in cases.items():
    scores = cosine_similarity(vectorizer.transform([text]), corpus)[0]
    nearest = scores.argsort()[::-1][:3]

    print(f"\n{name}\n{text}")
    for position in nearest:
        doc = docs.iloc[position]
        print(f"  {scores[position]:.3f} | {doc['id']} | {doc['title']}")
PY


Reporte escolar
Our school team delivered a history project and presented the final report.
  0.146 | PR02 | Sprint 14 Review - Logistics App
  0.125 | PR06 | Project Kickoff Minutes - Insurance CRM
  0.101 | PR05 | Cloud Migration Project Closure Report

Reporte de proyecto de TI
The integration project is two weeks behind schedule. The client approved a revised budget and a new delivery milestone.
  0.174 | PR01 | Retail Platform Project Status Report - March
  0.146 | PR02 | Sprint 14 Review - Logistics App
  0.135 | PR08 | Weekly Status Report - Data Warehouse

Capacitación
This onboarding lesson shows new team members how to deploy a service to Kubernetes for their first client project.
  0.194 | TD03 | Kubernetes Deployment Guide
  0.116 | TM10 | Leadership Program for Team Leads
  0.100 | PR03 | Lessons Learned - Banking Portal Project


### Comparación con documentos del corpus

El reporte escolar obtuvo una similitud máxima de 0.146 y el reporte de TI obtuvo 0.174. Ambos recuperaron reportes de proyecto y compartieron el documento PR02 entre sus tres primeros resultados. La diferencia observada es pequeña y estos ejemplos son insuficientes para fijar un umbral de pertenencia al dominio.

En la capacitación sobre Kubernetes, el primer resultado fue la guía técnica TD03 (0.194). Este caso ilustra cómo la similitud puede servir para sugerir material de consulta relacionado, incluso cuando el documento recomendado pertenece a otra categoría.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

MPLBACKEND=Agg PYTHONPATH=src .venv/bin/python - <<'PY'
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import FeatureUnion
from classifier import MODELS, SEED, build_pipeline, load_documents

docs = load_documents()
expected = docs["category"].to_numpy()
model = build_pipeline(MODELS["Logistic Regression"]())
model.set_params(tfidf=FeatureUnion([
    ("words", model["tfidf"]),
    ("chars", TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True
    )),
]))

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
probs = cross_val_predict(
    model, docs["content"], expected, cv=folds, method="predict_proba"
)
scores = probs.max(axis=1)
correct = np.unique(expected)[probs.argmax(axis=1)] == expected

rows = []
for threshold in (0, 0.30, 0.35, 0.40, 0.45, 0.50):
    accepted = scores >= threshold
    rows.append({
        "Umbral": f"{threshold:.0%}",
        "Aciertos aceptados": int((accepted & correct).sum()),
        "Errores aceptados": int((accepted & ~correct).sum()),
        "Total a revisión": int((~accepted).sum()),
        "Cobertura automática": f"{accepted.mean():.1%}",
    })

print(pd.DataFrame(rows).to_string(index=False))
PY

Umbral  Aciertos aceptados  Errores aceptados  Total a revisión Cobertura automática
    0%                  57                  3                 0               100.0%
   30%                  54                  3                 3                95.0%
   35%                  50                  2                 8                86.7%
   40%                  41                  1                18                70.0%
   45%                  29                  0                31                48.3%
   50%                  21                  0                39                35.0%


### Selección del modelo y del umbral

Seleccionamos el clasificador combinado de palabras y fragmentos de caracteres con un umbral provisional de 40%. En validación cruzada sobre 60 documentos, el sistema aceptó automáticamente 42 clasificaciones (70% de cobertura). De estas, 41 fueron correctas y una fue incorrecta. Los 18 documentos restantes pasaron a revisión.

Con un umbral de 45% se eliminaron los errores observados entre las clasificaciones aceptadas, pero la revisión aumentó de 18 a 31 documentos. Elegimos el 40% para reducir la carga de revisión, teniendo presente el error observado.

El umbral se seleccionó y comparó sobre este corpus. Su desempeño en documentos nuevos queda pendiente de validación. Además, la prueba del reporte escolar mostró que un texto fuera del dominio puede superar el umbral.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

git pull --ff-only origin main
printf 'Versión evaluada: '
git rev-parse --short HEAD

.venv/bin/python src/classifier.py

.venv/bin/python - <<'PY'
import pandas as pd

results = pd.read_csv("outputs/new_documents_classified.csv")
print("\nResultados del lote")
print(results[["id", "predicted_category", "confidence", "status"]].to_string(
    index=False,
    formatters={"confidence": "{:.2%}".format},
))
PY

Updating fae309a..e862b7f
Fast-forward
 src/classifier.py | 48 ++++++++++++++++++++++++++++++++++--------------
 1 file changed, 34 insertions(+), 14 deletions(-)
Versión evaluada: e862b7f
Loaded 60 documents in 5 categories:
category
Technical Documentation    12
Policies & Procedures      12
Project Reports            12
Training Manuals           12
Incident Reports           12 

Training documents: 45 | Test documents: 15

Naive Bayes: hold-out accuracy = 0.667 | 5-fold CV accuracy = 0.817 ± 0.082
                         precision    recall  f1-score   support

Technical Documentation       0.00      0.00      0.00         3
  Policies & Procedures       1.00      1.00      1.00         3
        Project Reports       0.75      1.00      0.86         3
       Training Manuals       0.75      1.00      0.86         3
       Incident Reports       0.33      0.33      0.33         3

               accuracy                           0.67        15
              macro avg       0.57 

From https://github.com/ierisaac2005-png/softtek-knowledge-hub
 * branch            fase-2     -> FETCH_HEAD
   fae309a..e862b7f  fase-2     -> origin/fase-2


### Verificación del clasificador integrado

Ejecutamos en Colab la versión `e862b7f` del clasificador de Knowledge Hub. El modelo combinado de palabras y caracteres clasificó correctamente 57 de 60 documentos en validación cruzada de cinco particiones (95%). En la prueba separada de 15 documentos acertó los 15.

Al procesar seis documentos nuevos con el umbral de 40%, el sistema aceptó cuatro clasificaciones y envió dos a revisión: N02 (39.02%) y N06 (38.46%). Para ambos conservó la categoría sugerida, el puntaje y el estado en `outputs/new_documents_classified.csv`.

N06 ilustra el cambio respecto al modelo original: antes se clasificaba como reporte de proyecto. La versión integrada lo identificó como manual de capacitación y lo marcó para revisión por su puntaje.

## Recomendación de documentos relacionados

Tomamos como punto de partida el reporte de incidente IR01 y recomendamos cinco documentos según la similitud de sus títulos y contenidos. Excluimos el propio IR01 para evitar que el sistema lo recomiende como resultado.

Para evaluar esta primera versión, comprobamos en qué posición aparecen dos documentos vinculados al incidente en la ontología: TD01, citado directamente por el reporte, y TD07, relacionado mediante la lección aprendida.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

PYTHONPATH=src .venv/bin/python - <<'PY'
from sklearn.metrics.pairwise import cosine_similarity
from kms_utils import load_documents
from semantic_search import SemanticSearch

docs = load_documents().reset_index(drop=True)
engine = SemanticSearch(docs)

source_id = "IR01"
source_index = docs.index[docs["id"] == source_id][0]
scores = cosine_similarity(
    engine.doc_matrix[source_index], engine.doc_matrix
)[0]

ranking = docs[["id", "title", "category"]].copy()
ranking["similarity"] = scores
ranking = (
    ranking.loc[ranking["id"] != source_id]
    .sort_values(["similarity", "id"], ascending=[False, True])
    .reset_index(drop=True)
)
ranking.index += 1
ranking.index.name = "Posición"

print(f"Documento consultado: {source_id} — {docs.at[source_index, 'title']}")
print("\nCinco recomendaciones por similitud")
print(ranking.head(5).to_string(
    formatters={"similarity": "{:.3f}".format}
))

print("\nPosición de los documentos vinculados en la ontología")
for doc_id, relation in [
    ("TD01", "Referencia directa del reporte"),
    ("TD07", "Guía vinculada mediante la lección aprendida"),
]:
    position = ranking.index[ranking["id"] == doc_id][0]
    print(f"{doc_id} | posición {position} de {len(ranking)} | {relation}")
PY

Documento consultado: IR01 — Post-mortem: Payments API Outage

Cinco recomendaciones por similitud
            id                                             title                 category similarity
Posición                                                                                            
1         IR10    Post-mortem: Cache Stampede on Product Catalog         Incident Reports      0.155
2         TD01                         Payments API Reference v3  Technical Documentation      0.109
3         IR07                Post-mortem: Message Queue Backlog         Incident Reports      0.090
4         IR05         Post-mortem: Failed Production Deployment         Incident Reports      0.088
5         IR12  Root Cause Analysis: Disk Full on Logging Server         Incident Reports      0.063

Posición de los documentos vinculados en la ontología
TD01 | posición 2 de 59 | Referencia directa del reporte
TD07 | posición 25 de 59 | Guía vinculada mediante la lección aprendida


### Incorporación de la ontología al recomendador

Para IR01, el recomendador por similitud colocó la referencia técnica TD01 en la posición 2 de 59 y la guía de monitoreo TD07 en la posición 25. Cuatro de las primeras cinco recomendaciones fueron otros reportes de incidentes.

Probamos una segunda versión que prioriza las referencias directas y los documentos vinculados mediante lecciones aprendidas. Las posiciones restantes se completan por similitud textual. Cada resultado muestra el motivo de su inclusión y la posición que ocupaba en la versión original.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

PYTHONPATH=src .venv/bin/python - <<'PY'
from rdflib import Graph, Literal, Namespace
from sklearn.metrics.pairwise import cosine_similarity
from kms_utils import ONTOLOGY_PATH, load_documents
from semantic_search import SemanticSearch

source_id = "IR01"
docs = load_documents().reset_index(drop=True)
source_index = docs.index[docs["id"] == source_id][0]

engine = SemanticSearch(docs)
scores = cosine_similarity(engine.doc_matrix[source_index], engine.doc_matrix)[0]
ranking = docs[["id", "title"]].copy()
ranking["similitud"] = scores
ranking = (
    ranking.loc[ranking["id"] != source_id]
    .sort_values(["similitud", "id"], ascending=[False, True])
    .reset_index(drop=True)
)
ranking["posición original"] = ranking.index + 1

graph = Graph().parse(ONTOLOGY_PATH, format="turtle")
ex = Namespace("http://softtek.example/kms#")

# El orden de las rutas define la prioridad si hay más de un motivo.
routes = [
    (ex.references, "Referencia directa"),
    (ex.documents / ex.resultsIn / ex.updates, "Lección aprendida"),
]
links = {}
for source in graph.subjects(ex.docId, Literal(source_id)):
    for priority, (route, reason) in enumerate(routes):
        for target in graph.objects(source, route):
            for doc_id in graph.objects(target, ex.docId):
                key = str(doc_id)
                if key not in links or priority < links[key][0]:
                    links[key] = (priority, reason)

fallback = (len(routes), "Similitud textual")
ranking["prioridad"] = [links.get(key, fallback)[0] for key in ranking["id"]]
ranking["motivo"] = [links.get(key, fallback)[1] for key in ranking["id"]]
top = ranking.sort_values(["prioridad", "posición original"]).head(5).copy()
top.index = range(1, len(top) + 1)
top.index.name = "Posición final"

print(f"Recomendaciones para {source_id} — {docs.at[source_index, 'title']}\n")
print(top[["id", "title", "motivo", "posición original", "similitud"]].to_string(
    formatters={"similitud": "{:.3f}".format}
))
PY

Recomendaciones para IR01 — Post-mortem: Payments API Outage

                  id                                           title              motivo  posición original similitud
Posición final                                                                                                       
1               TD01                       Payments API Reference v3  Referencia directa                  2     0.109
2               TD07                    Logging and Monitoring Setup   Lección aprendida                 25     0.011
3               IR10  Post-mortem: Cache Stampede on Product Catalog   Similitud textual                  1     0.155
4               IR07              Post-mortem: Message Queue Backlog   Similitud textual                  3     0.090
5               IR05       Post-mortem: Failed Production Deployment   Similitud textual                  4     0.088


### Resultado del recomendador híbrido para IR01

El recomendador recuperó dos lecturas con una relación documentada con el incidente: la referencia de la API de pagos (TD01) y la guía de monitoreo (TD07). Esta última pasó de la posición 25 a la 2, a pesar de tener una similitud textual de solo 0.011. Su incorporación desplazó del top 5 al reporte IR12, sobre un servidor de logs sin espacio.

El resultado muestra el motivo de cada sugerencia y permite distinguir una referencia documentada de una coincidencia textual. En este caso se comprobó que el recomendador sigue las relaciones definidas en la ontología.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

PYTHONPATH=src .venv/bin/python - <<'PY'
from rdflib import Graph, Literal, Namespace
from sklearn.metrics.pairwise import cosine_similarity
from kms_utils import ONTOLOGY_PATH, load_documents
from semantic_search import SemanticSearch

docs = load_documents().reset_index(drop=True)
engine = SemanticSearch(docs)
graph = Graph().parse(ONTOLOGY_PATH, format="turtle")
ex = Namespace("http://softtek.example/kms#")

routes = [
    (ex.references, "Referencia directa"),
    (ex.documents / ex.resultsIn / ex.updates, "Lección aprendida"),
]

def recommend(source_id, k=5):
    source_index = docs.index[docs["id"] == source_id][0]
    scores = cosine_similarity(engine.doc_matrix[source_index], engine.doc_matrix)[0]
    ranking = docs[["id", "title"]].assign(similitud=scores)
    ranking = ranking.loc[ranking["id"] != source_id].copy()

    links = {}
    for priority, (route, reason) in enumerate(routes):
        for source in graph.subjects(ex.docId, Literal(source_id)):
            for target in graph.objects(source, route):
                for doc_id in graph.objects(target, ex.docId):
                    links.setdefault(str(doc_id), (priority, reason))

    fallback = (len(routes), "Similitud textual")
    ranking["prioridad"] = [links.get(key, fallback)[0] for key in ranking["id"]]
    ranking["motivo"] = [links.get(key, fallback)[1] for key in ranking["id"]]
    return ranking.sort_values(
        ["prioridad", "similitud", "id"], ascending=[True, False, True]
    ).head(k)

for source_id in ["IR03", "TM03", "IR07"]:
    top = recommend(source_id)
    print(f"\nRecomendaciones para {source_id}")
    print(top[["id", "title", "motivo"]].to_string(index=False))

    assert len(top) == 5, f"{source_id}: se esperaban cinco recomendaciones"
    assert top["id"].is_unique, f"{source_id}: hay recomendaciones duplicadas"
    assert source_id not in top["id"].values, f"{source_id}: se recomendó a sí mismo"

    reasons = top.set_index("id")["motivo"].to_dict()
    if source_id == "IR03":
        assert reasons.get("TD04") == "Referencia directa"
        assert reasons.get("PP07") == "Lección aprendida"
    elif source_id == "TM03":
        assert reasons.get("TD03") == "Referencia directa"
    else:
        assert top["motivo"].eq("Similitud textual").all()

print("\nLas tres pruebas funcionales pasaron.")
PY


Recomendaciones para IR03
  id                                            title             motivo
TD04            Database Schema for Inventory Service Referencia directa
PP07                      Change Management Procedure  Lección aprendida
IR07               Post-mortem: Message Queue Backlog  Similitud textual
IR12 Root Cause Analysis: Disk Full on Logging Server  Similitud textual
IR10   Post-mortem: Cache Stampede on Product Catalog  Similitud textual

Recomendaciones para TM03
  id                                     title             motivo
TD03               Kubernetes Deployment Guide Referencia directa
IR05 Post-mortem: Failed Production Deployment  Similitud textual
TM12   Information Security Awareness Training  Similitud textual
TM09          Cloud Practitioner Learning Path  Similitud textual
TD02       Microservices Architecture Overview  Similitud textual

Recomendaciones para IR07
  id                                                 title            motivo
TD12    

### Verificación funcional del recomendador híbrido

En IR03, el sistema recomendó TD04 una sola vez y mostró la referencia directa como motivo, aunque también podía llegar al documento mediante una lección aprendida. Por esa segunda ruta recuperó PP07, el procedimiento de gestión de cambios.

En TM03, la primera recomendación fue TD03, la guía de despliegue de Kubernetes citada por el manual. En IR07, que carece de vínculos explícitos en la ontología, la similitud textual recuperó TD12, una guía técnica relacionada con el incidente.

En los tres casos se obtuvieron cinco documentos distintos y se excluyó el documento consultado. Estas pruebas verifican el funcionamiento de las reglas. La relevancia del conjunto de recomendaciones queda pendiente para la evaluación de métricas.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

git pull --ff-only origin main
printf 'Versión evaluada: '
git rev-parse --short HEAD

for documento in IR01 IR03 TM03 IR07; do
  printf '\nRecomendaciones para %s\n' "$documento"
  .venv/bin/python src/recommender.py "$documento"
done

Updating e862b7f..2ab47be
Fast-forward
 src/recommender.py | 66 ++++++++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 66 insertions(+)
 create mode 100644 src/recommender.py
Versión evaluada: 2ab47be

Recomendaciones para IR01
  id                                          title                category similarity             reason
TD01                      Payments API Reference v3 Technical Documentation      0.109 Referencia directa
TD07                   Logging and Monitoring Setup Technical Documentation      0.011  Lección aprendida
IR10 Post-mortem: Cache Stampede on Product Catalog        Incident Reports      0.155  Similitud textual
IR07             Post-mortem: Message Queue Backlog        Incident Reports      0.090  Similitud textual
IR05      Post-mortem: Failed Production Deployment        Incident Reports      0.088  Similitud textual

Recomendaciones para IR03
  id                                            title                category similarity    

From https://github.com/ierisaac2005-png/softtek-knowledge-hub
 * branch            fase-2     -> FETCH_HEAD
   e862b7f..2ab47be  fase-2     -> origin/fase-2


### Verificación del módulo integrado

Ejecutamos `src/recommender.py` desde el commit `2ab47be` con los documentos IR01, IR03, TM03 e IR07. El módulo reprodujo los resultados del prototipo y mostró el motivo de cada recomendación. En IR03 recuperó el procedimiento PP07 mediante una lección aprendida. En IR07 encontró la guía técnica TD12 por similitud textual.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

PYTHONPATH=src .venv/bin/python - <<'PY'
from kms_utils import load_documents
from semantic_search import SemanticSearch

engine = SemanticSearch(load_documents())
queries = [
    ("Consulta original", "what happened when the payment system went down"),
    ("Reformulación en inglés", "Why were customers unable to complete their purchases?"),
    ("Pregunta en español", "¿Por qué los clientes no pudieron completar sus compras?"),
]

for label, query in queries:
    print(f"\n{label}: {query}")
    results = engine.search(query, top_k=5).reset_index(drop=True)

    if results.empty:
        print("Sin resultados.")
        continue

    results.index += 1
    results.index.name = "Posición"
    print(results[["id", "title", "similarity"]].to_string())
    positions = results.index[results["id"] == "IR01"].tolist()
    print(f"IR01 quedó en la posición {positions[0]}." if positions
          else "IR01 quedó fuera de los primeros cinco.")
PY


Consulta original: what happened when the payment system went down
            id                                              title  similarity
Posición                                                                     
1         IR01                   Post-mortem: Payments API Outage       0.197
2         IR11  Incident Report: Third-Party Email Service Outage       0.195
3         IR04       Incident Report: Data Center Network Failure       0.168
4         IR06              Incident: Unauthorized Access Attempt       0.090
5         IR10     Post-mortem: Cache Stampede on Product Catalog       0.066
IR01 quedó en la posición 1.

Reformulación en inglés: Why were customers unable to complete their purchases?
            id                                       title  similarity
Posición                                                              
1         IR01            Post-mortem: Payments API Outage       0.247
2         TM06        Customer Communication Skills Course     

### Efecto del idioma en una consulta sobre el mismo incidente

El buscador recuperó IR01 en primer lugar con las dos consultas en inglés. Al formular la pregunta equivalente en español, colocó primero a PR09, una encuesta de satisfacción, e IR01 quedó fuera de los cinco resultados. En este caso, el buscador produjo una lista de coincidencias que no respondía a la pregunta.

Para determinar si el problema se repite en otras categorías, evaluamos pares de preguntas en inglés y español con un documento esperado definido antes de la ejecución.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

PYTHONPATH=src .venv/bin/python - <<'PY'
import pandas as pd
from kms_utils import load_documents
from semantic_search import SemanticSearch

engine = SemanticSearch(load_documents())

cases = [
    ("Incidentes", "IR01",
     "Why were customers unable to complete their purchases?",
     "¿Por qué los clientes no pudieron completar sus compras?"),
    ("Políticas", "PP02",
     "What is the company policy on working from home?",
     "¿Cuál es la política de la empresa sobre el trabajo desde casa?"),
    ("Documentación técnica", "TD03",
     "Where is the guide for deploying a service to Kubernetes?",
     "¿Dónde está la guía para desplegar un servicio en Kubernetes?"),
    ("Capacitación", "TM12",
     "Which training course teaches employees to recognize phishing emails?",
     "¿Qué curso enseña a los empleados a reconocer correos de phishing?"),
    ("Reportes de proyecto", "PR10",
     "Why did the mobile banking project exceed its budget?",
     "¿Por qué el proyecto de banca móvil superó su presupuesto?"),
]

rows = []
for category, expected, english, spanish in cases:
    for language, query in [("EN", english), ("ES", spanish)]:
        results = engine.search(query, top_k=5)
        ids = results["id"].tolist()
        rank = ids.index(expected) + 1 if expected in ids else None
        rows.append({
            "Categoría": category,
            "Idioma": language,
            "Esperado": expected,
            "Primero": ids[0] if ids else "Sin resultados",
            "Posición": str(rank) if rank else "Fuera del top 5",
            "Top 1": rank == 1,
            "Top 5": rank is not None,
        })
    terms = ", ".join(engine.query_terms(spanish)) or "ninguno"
    print(f"Términos reconocidos en ES · {category}: {terms}")

table = pd.DataFrame(rows)
print("\nResultados por pregunta")
print(table.iloc[:, :5].to_string(index=False))

print("\nConsultas que recuperaron el documento esperado, de 5 por idioma")
print(table.groupby("Idioma")[["Top 1", "Top 5"]].sum().to_string())
PY

Términos reconocidos en ES · Incidentes: client
Términos reconocidos en ES · Políticas: ninguno
Términos reconocidos en ES · Documentación técnica: kubernet
Términos reconocidos en ES · Capacitación: phish
Términos reconocidos en ES · Reportes de proyecto: ninguno

Resultados por pregunta
            Categoría Idioma Esperado        Primero        Posición
           Incidentes     EN     IR01           IR01               1
           Incidentes     ES     IR01           PR09 Fuera del top 5
            Políticas     EN     PP02           PP02               1
            Políticas     ES     PP02 Sin resultados Fuera del top 5
Documentación técnica     EN     TD03           TD03               1
Documentación técnica     ES     TD03           TD03               1
         Capacitación     EN     TM12           TM12               1
         Capacitación     ES     TM12           IR02               2
 Reportes de proyecto     EN     PR10           PR10               1
 Reportes de proyect

### Diagnóstico de la búsqueda bilingüe

En cinco pares de consultas equivalentes, el buscador recuperó el documento esperado en primer lugar en 5/5 consultas en inglés y 1/5 en español. En español, dos preguntas no devolvieron resultados. En otras, el sistema reconoció únicamente un término compartido, como Kubernetes o phishing.

Como primera mejora, probamos ampliar la consulta en español con términos equivalentes en inglés antes de buscar en el corpus. Comparamos el resultado con la búsqueda original y registramos qué traducciones se aplicaron.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

PYTHONPATH=src .venv/bin/python - <<'PY'
import re
import unicodedata
import pandas as pd
from kms_utils import load_documents
from semantic_search import SemanticSearch

engine = SemanticSearch(load_documents())

glossary = {
    "clientes": "customers",
    "compras": "purchases",
    "política": "policy",
    "trabajo desde casa": "remote work",
    "guía": "guide",
    "desplegar": "deploy",
    "servicio": "service",
    "curso": "training",
    "empleados": "employees",
    "correos": "email",
    "proyecto": "project",
    "banca móvil": "mobile banking",
    "presupuesto": "budget",
}

def normalize(text):
    decomposed = unicodedata.normalize("NFD", text.casefold())
    return "".join(char for char in decomposed if not unicodedata.combining(char))

def expand_spanish(query):
    normalized = normalize(query)
    additions = [
        english for spanish, english in glossary.items()
        if re.search(rf"\b{re.escape(normalize(spanish))}\b", normalized)
    ]
    return " ".join([query, *additions]), additions

def position(query, expected):
    ids = engine.search(query, top_k=5)["id"].tolist()
    return ids.index(expected) + 1 if expected in ids else None

cases = [
    ("Inicial", "IR01", "¿Por qué los clientes no pudieron completar sus compras?"),
    ("Inicial", "PP02", "¿Cuál es la política de la empresa sobre el trabajo desde casa?"),
    ("Inicial", "TD03", "¿Dónde está la guía para desplegar un servicio en Kubernetes?"),
    ("Inicial", "TM12", "¿Qué curso enseña a los empleados a reconocer correos de phishing?"),
    ("Inicial", "PR10", "¿Por qué el proyecto de banca móvil superó su presupuesto?"),
    ("Adicional", "TM02", "¿Qué curso explica cómo usar Git para control de versiones?"),
    ("Adicional", "PP08", "¿Cómo puedo solicitar vacaciones?"),
]

rows = []
for group, expected, query in cases:
    expanded, additions = expand_spanish(query)
    before = position(query, expected)
    after = position(expanded, expected)
    rows.append({
        "Grupo": group,
        "Esperado": expected,
        "Antes": before or "Fuera del top 5",
        "Después": after or "Fuera del top 5",
        "Términos añadidos": ", ".join(additions) or "Ninguno",
    })

table = pd.DataFrame(rows)
print(table.to_string(index=False))

for group, subset in table.groupby("Grupo", sort=False):
    print(f"\n{group} · documento esperado en primera posición")
    for column in ["Antes", "Después"]:
        print(f"  {column}: {subset[column].eq(1).sum()}/{len(subset)}")
PY

    Grupo Esperado           Antes         Después               Términos añadidos
  Inicial     IR01 Fuera del top 5               1            customers, purchases
  Inicial     PP02 Fuera del top 5               1             policy, remote work
  Inicial     TD03               1               1          guide, deploy, service
  Inicial     TM12               2               1      training, employees, email
  Inicial     PR10 Fuera del top 5               1 project, mobile banking, budget
Adicional     TM02               2               4                        training
Adicional     PP08 Fuera del top 5 Fuera del top 5                         Ninguno

Inicial · documento esperado en primera posición
  Antes: 1/5
  Después: 5/5

Adicional · documento esperado en primera posición
  Antes: 0/2
  Después: 0/2


### Decisión sobre el glosario bilingüe

La expansión mejoró de 1/5 a 5/5 los primeros resultados en las consultas que guiaron su diseño. En las dos consultas adicionales, el documento sobre Git pasó de la posición 2 a la 4 y la política de vacaciones siguió fuera del top 5. Por ello, dejamos el glosario como experimento y conservamos el buscador actual como referencia.

El siguiente experimento comparará TF-IDF con recuperación multilingüe sobre preguntas y documentos relevantes definidos antes de evaluar los modelos.

## Conjunto de evaluación para la búsqueda en español

Definimos diez consultas en español y asignamos a cada una un documento de referencia del corpus en inglés. Las consultas cubren las cinco categorías y se fijan antes de comparar los buscadores. Evaluaremos cuántas veces aparece el documento de referencia en primer lugar y entre los primeros cinco resultados. También revisaremos en qué preguntas mejora o empeora su posición.

In [ ]:
from pathlib import Path
import pandas as pd

repo = Path("/content/softtek-knowledge-hub")
corpus = pd.read_csv(repo / "data/documents.csv")

cases = [
    ("ES01", "¿En qué tablas se guardan los productos, los almacenes y los movimientos de existencias?", "TD04"),
    ("ES02", "¿Cómo se entera el equipo de guardia cuando las aplicaciones tardan demasiado en responder?", "TD07"),
    ("ES03", "¿Qué plazo tiene la empresa para responder a alguien que pregunta por el uso de sus datos personales?", "PP04"),
    ("ES04", "¿Cuántos años debemos guardar las facturas y qué pasa si hay un litigio pendiente?", "PP10"),
    ("ES05", "¿Cuánto bajó el gasto de infraestructura después de trasladar las aplicaciones del cliente a la nube?", "PR05"),
    ("ES06", "¿Qué acordó cambiar el equipo de pagos después de calcular mal el esfuerzo de las tareas del sprint?", "PR11"),
    ("ES07", "¿Dónde puedo practicar cómo explicar resultados técnicos a un cliente sin experiencia en tecnología?", "TM06"),
    ("ES08", "Soy líder de equipo y quiero armar un plan de desarrollo personal con ayuda de un mentor. ¿Qué programa puedo tomar?", "TM10"),    ("ES09", "¿Por qué los pedidos se procesaron con tres horas de retraso?", "IR07"),
    ("ES10", "¿Por qué dejamos de recibir registros cuando se llenó el disco y por qué la alerta llegó tarde?", "IR12"),
]

evaluation = pd.DataFrame(cases, columns=["query_id", "query_es", "expected_id"])
assert evaluation["query_id"].is_unique
assert evaluation["expected_id"].isin(corpus["id"]).all()

review = evaluation.merge(
    corpus[["id", "title", "category"]],
    left_on="expected_id",
    right_on="id",
    validate="one_to_one",
)
assert len(review) == 10 and review["category"].value_counts().eq(2).all()

output = repo / "data/evaluacion_busqueda_es.csv"
evaluation.to_csv(output, index=False)

print(f"Guardado: {output.name} | {len(review)} consultas\n")
for row in review.itertuples():
    print(f"{row.query_id} · {row.query_es}")
    print(f"     {row.expected_id} — {row.title}\n")

Guardado: evaluacion_busqueda_es.csv | 10 consultas

ES01 · ¿En qué tablas se guardan los productos, los almacenes y los movimientos de existencias?
     TD04 — Database Schema for Inventory Service

ES02 · ¿Cómo se entera el equipo de guardia cuando las aplicaciones tardan demasiado en responder?
     TD07 — Logging and Monitoring Setup

ES03 · ¿Qué plazo tiene la empresa para responder a alguien que pregunta por el uso de sus datos personales?
     PP04 — Data Privacy and Personal Data Protection Policy

ES04 · ¿Cuántos años debemos guardar las facturas y qué pasa si hay un litigio pendiente?
     PP10 — Document Retention Policy

ES05 · ¿Cuánto bajó el gasto de infraestructura después de trasladar las aplicaciones del cliente a la nube?
     PR05 — Cloud Migration Project Closure Report

ES06 · ¿Qué acordó cambiar el equipo de pagos después de calcular mal el esfuerzo de las tareas del sprint?
     PR11 — Sprint 22 Retrospective - Payments Team

ES07 · ¿Dónde puedo practicar cómo ex

### Comparación de TF-IDF y búsqueda multilingüe

Evaluamos los dos buscadores sobre las diez preguntas en español definidas previamente. Ambos reciben el mismo texto y buscan entre los mismos documentos en inglés, utilizando el título y el contenido. Para cada pregunta registramos el primer resultado y la posición del documento esperado dentro de los cinco primeros. Al final comparamos los aciertos Top 1 y Top 5 de cada método.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

python -m pip --python .venv/bin/python install -q \
  --index-url https://download.pytorch.org/whl/cpu torch

python -m pip --python .venv/bin/python install -q \
  -r requirements.txt sentence-transformers

PYTHONPATH=src .venv/bin/python - <<'PY'
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from kms_utils import load_documents
from semantic_search import SemanticSearch

docs = load_documents().reset_index(drop=True)
cases = pd.read_csv("data/evaluacion_busqueda_es.csv")
assert len(cases) == 10 and cases["query_id"].is_unique
assert cases["expected_id"].isin(docs["id"]).all()

lexical = SemanticSearch(docs)
model_name = "intfloat/multilingual-e5-small"
encoder = SentenceTransformer(model_name, device="cpu")

passages = ["passage: " + text for text in docs["content"]]
queries = ["query: " + text for text in cases["query_es"]]

doc_vectors = encoder.encode(passages, normalize_embeddings=True)
query_vectors = encoder.encode(queries, normalize_embeddings=True)
similarities = query_vectors @ doc_vectors.T

rows = []
for i, case in enumerate(cases.itertuples(index=False)):
    lexical_ids = lexical.search(case.query_es, top_k=5)["id"].tolist()

    best_indices = np.argsort(-similarities[i], kind="stable")[:5]
    multilingual_ids = docs.iloc[best_indices]["id"].tolist()

    for method, ids in [("TF-IDF", lexical_ids), ("E5", multilingual_ids)]:
        position = ids.index(case.expected_id) + 1 if case.expected_id in ids else None
        rows.append({
            "consulta": case.query_id,
            "esperado": case.expected_id,
            "método": method,
            "primero": ids[0] if ids else "Sin resultados",
            "posición": position,
            "top_5_ids": ", ".join(ids),
        })

results = pd.DataFrame(rows)
results["posición"] = results["posición"].astype("Int64")

comparison = results.pivot(
    index=["consulta", "esperado"],
    columns="método",
    values=["primero", "posición"],
)
print("\nResultados por consulta (— indica fuera del top 5)")
print(comparison.to_string(na_rep="—"))

print("\nResumen sobre 10 consultas")
for method, group in results.groupby("método", sort=False):
    top1 = group["posición"].eq(1).sum()
    top5 = group["posición"].notna().sum()
    print(f"{method:7}  Top 1: {top1}/10   Top 5: {top5}/10")

output = Path("outputs/comparacion_busqueda_es.csv")
output.parent.mkdir(exist_ok=True)
results.to_csv(output, index=False)
print(f"\nEvidencia guardada en {output}")
PY


Resultados por consulta (— indica fuera del top 5)
                  primero                 posición       
método                 E5          TF-IDF       E5 TF-IDF
consulta esperado                                        
ES01     TD04        TD04  Sin resultados        1   <NA>
ES02     TD07        IR10  Sin resultados     <NA>   <NA>
ES03     PP04        PP04  Sin resultados        1   <NA>
ES04     PP10        PP10  Sin resultados        1   <NA>
ES05     PR05        PR05            PR09        1   <NA>
ES06     PR11        PR11            PR02        1      2
ES07     TM06        TM06            PR09        1   <NA>
ES08     TM10        TM10            TM10        1      1
ES09     IR07        IR07  Sin resultados        1   <NA>
ES10     IR12        IR12  Sin resultados        1   <NA>

Resumen sobre 10 consultas
TF-IDF   Top 1: 1/10   Top 5: 2/10
E5       Top 1: 9/10   Top 5: 9/10

Evidencia guardada en outputs/comparacion_busqueda_es.csv


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8387.26it/s]


### Resultado de la evaluación multilingüe

Sobre las diez consultas en español, TF-IDF recuperó el documento esperado en primer lugar en 1/10 casos y dentro de los cinco primeros en 2/10. E5 alcanzó 9/10 en ambas métricas. En ocho consultas mejoró la primera posición y en ES08 ambos métodos recuperaron el documento correcto.

La excepción fue ES02, sobre cómo se avisa al equipo de guardia cuando aumenta la latencia. E5 colocó primero a IR10 y el documento de referencia TD07 quedó fuera de los cinco primeros. Seleccionamos E5 como candidato para las consultas en español y revisamos este caso antes de integrarlo.

In [ ]:
from pathlib import Path
import pandas as pd

repo = Path("/content/softtek-knowledge-hub")
results = pd.read_csv(repo / "outputs/comparacion_busqueda_es.csv")
cases = pd.read_csv(repo / "data/evaluacion_busqueda_es.csv").set_index("query_id")
docs = pd.read_csv(repo / "data/documents.csv").set_index("id")

case = cases.loc["ES02"]
result = results.loc[
    (results["consulta"] == "ES02") & (results["método"] == "E5")
].iloc[0]

print("PREGUNTA:", case["query_es"])

def show_document(label, doc_id):
    doc = docs.loc[doc_id]
    print(f"\n{label} · {doc_id} · {doc['title']}")
    print(doc["text"])

show_document("Referencia esperada", case["expected_id"])

for rank, doc_id in enumerate(result["top_5_ids"].split(", "), start=1):
    show_document(f"Resultado {rank}", doc_id)

PREGUNTA: ¿Cómo se entera el equipo de guardia cuando las aplicaciones tardan demasiado en responder?

Referencia esperada · TD07 · Logging and Monitoring Setup
Applications write structured JSON logs that are shipped to Elasticsearch through Fluent Bit. Metrics are exposed in Prometheus format and visualized in Grafana dashboards. Alert rules notify the on-call channel when latency or error rate exceed thresholds.

Resultado 1 · IR10 · Post-mortem: Cache Stampede on Product Catalog
When the cache cluster restarted all requests went to the database at the same time, causing timeouts. The service was degraded for thirty minutes. Corrective actions include request coalescing and staggered cache expiration.

Resultado 2 · IR07 · Post-mortem: Message Queue Backlog
Order events accumulated in the queue and were processed with a delay of three hours. The root cause was a consumer that crashed due to an unhandled malformed message. A fix was deployed and monitoring of queue depth was added to

### Análisis del error ES02

La revisión del contenido confirmó que TD07 responde directamente a la pregunta. Su última oración identifica la condición que dispara la alerta y el canal que recibe la notificación. Los cinco resultados recuperados por E5 contienen coincidencias parciales relacionadas con lentitud, incidentes o alertas, pero ninguno explica ambos elementos. Conservamos ES02 como fallo en la evaluación original: Top 1 y Top 5 permanecen en 9/10.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

PYTHONPATH=src .venv/bin/python - <<'PY'
import re
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

from kms_utils import load_documents

docs = load_documents()
cases = pd.read_csv("data/evaluacion_busqueda_es.csv")
previous = pd.read_csv("outputs/comparacion_busqueda_es.csv")
previous = previous.loc[previous["método"] == "E5"].set_index("consulta")

# Conservamos el título como contexto y el ID para agrupar por documento.
fragments = []
for doc in docs.itertuples():
    for sentence in re.split(r"(?<=[.!?])\s+", doc.text.strip()):
        fragments.append({
            "id": doc.id,
            "oración": sentence,
            "entrada": f"passage: {doc.title}. {sentence}",
        })

fragments = pd.DataFrame(fragments)
model = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
passages = model.encode(fragments["entrada"].tolist(), normalize_embeddings=True)
queries = model.encode(
    ["query: " + question for question in cases["query_es"]],
    normalize_embeddings=True,
)

rows = []
for case, query in zip(cases.itertuples(index=False), queries):
    # Cada documento participa una sola vez, con su oración mejor puntuada.
    ranked = (
        fragments.assign(puntaje=passages @ query)
        .sort_values("puntaje", ascending=False, kind="stable")
        .drop_duplicates("id")
        .reset_index(drop=True)
    )
    top_ids = ranked.head(5)["id"].tolist()
    new_rank = top_ids.index(case.expected_id) + 1 if case.expected_id in top_ids else None
    rows.append({
        "consulta": case.query_id,
        "esperado": case.expected_id,
        "primero_anterior": previous.at[case.query_id, "primero"],
        "primero_fragmentos": top_ids[0],
        "posición_anterior": previous.at[case.query_id, "posición"],
        "posición_fragmentos": new_rank,
    })

    if case.query_id == "ES02":
        print("ES02 · Primeros cinco documentos y su mejor oración")
        print(ranked.head(5)[["id", "oración"]].to_string(index=False))
        expected_index = ranked.index[ranked["id"] == case.expected_id][0]
        print(f"\nPosición de TD07 entre los {len(ranked)} documentos: {expected_index + 1}")
        print("Mejor oración de TD07:", ranked.at[expected_index, "oración"])

results = pd.DataFrame(rows)
rank_columns = ["posición_anterior", "posición_fragmentos"]
results[rank_columns] = results[rank_columns].astype("Int64")

print("\nComparación sobre las mismas diez consultas")
print(results.to_string(index=False))

for column in rank_columns:
    ranks = results[column]
    print(f"{column}: Top 1 = {ranks.eq(1).sum()}/10 | Top 5 = {ranks.notna().sum()}/10")

output = Path("outputs/comparacion_e5_fragmentos_es.csv")
results.to_csv(output, index=False)
print(f"\nGuardado en {output}")
PY

ES02 · Primeros cinco documentos y su mejor oración
  id                                                                                                   oración
IR05                    The on-call engineer detected the alert and performed a rollback after twenty minutes.
IR07                                 A fix was deployed and monitoring of queue depth was added to the alerts.
IR10    When the cache cluster restarted all requests went to the database at the same time, causing timeouts.
TD12 Consumers must acknowledge messages after processing and failed messages are sent to a dead letter queue.
TM01                                                                                       Welcome to Softtek.

Posición de TD07 entre los 60 documentos: 12
Mejor oración de TD07: Alert rules notify the on-call channel when latency or error rate exceed thresholds.

Comparación sobre las mismas diez consultas
consulta esperado primero_anterior primero_fragmentos  posición_anterior  posición_

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6119.03it/s]


### Decisión sobre la unidad de búsqueda

Seleccionamos E5 sobre documentos completos para ordenar los resultados de consultas en español. En las diez preguntas evaluadas obtuvo 9/10 en Top 1 y 9/10 en Top 5. Al ordenar por la mejor oración de cada documento, el Top 1 bajó a 8/10 y el Top 5 permaneció en 9/10.

En ES02, la búsqueda por fragmentos identificó la oración pertinente dentro de TD07, pero ubicó el documento en la posición 12. En ES09, el documento correcto pasó del primer lugar al tercero. Conservamos ES02 como una limitación conocida y dejamos la selección de oraciones como posible apoyo para mostrar evidencia en el chatbot.

In [ ]:
import os
import subprocess
from pathlib import Path

pregunta = "¿Qué debo hacer para pedir días de vacaciones?"  # @param {type:"string"}
repo = Path("/content/softtek-knowledge-hub")

busqueda = """
import sys
from sentence_transformers import SentenceTransformer
from kms_utils import load_documents

pregunta = sys.argv[1].strip()
if not pregunta:
    raise SystemExit("Escribe una pregunta para buscar.")

docs = load_documents().reset_index(drop=True)
modelo = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")

vectores = modelo.encode(
    ["passage: " + texto for texto in docs["content"]],
    normalize_embeddings=True,
    show_progress_bar=False,
)
consulta = modelo.encode("query: " + pregunta, normalize_embeddings=True)
top = docs.assign(similitud=vectores @ consulta).nlargest(5, "similitud")

print("Pregunta:", pregunta)
print()
for posicion, doc in enumerate(top.itertuples(), start=1):
    print(f"{posicion}. {doc.id} | {doc.title} | {doc.similitud:.3f}")

primero = top.iloc[0]
print()
print(f"Contenido del primer resultado ({primero['id']}):")
print(primero["text"])
"""

resultado = subprocess.run(
    [str(repo / ".venv/bin/python"), "-c", busqueda, pregunta],
    cwd=repo,
    env={**os.environ, "PYTHONPATH": str(repo / "src")},
    capture_output=True,
    text=True,
)

if resultado.returncode != 0:
    raise RuntimeError(resultado.stderr.strip())

print(resultado.stdout.strip())

Pregunta: ¿Qué debo hacer para pedir días de vacaciones?

1. PP08 | Vacation and Leave Policy | 0.837
2. PP03 | Travel and Expense Reimbursement Procedure | 0.784
3. TM01 | New Employee Onboarding Guide | 0.782
4. PP02 | Remote Work Policy | 0.773
5. PP04 | Data Privacy and Personal Data Protection Policy | 0.762

Contenido del primer resultado (PP08):
Employees are entitled to vacation days according to their seniority and local labor law. Leave requests must be submitted through the HR portal at least two weeks in advance. Managers approve requests considering project deadlines and team coverage.


### Prueba de consulta libre en español

Al preguntar «¿Qué debo hacer para pedir días de vacaciones?», E5 recuperó en primer lugar `PP08 — Vacation and Leave Policy`, con una similitud de 0.837. El contenido permitió identificar el canal de solicitud, el plazo y quién la aprueba. En una formulación anterior sobre el mismo tema, el buscador con glosario había dejado PP08 fuera de sus primeros cinco resultados.

Esta prueba complementa la evaluación de diez consultas. El tema de vacaciones ya se había explorado y se documenta por separado.

### Integración de la búsqueda multilingüe

El buscador recibe varias preguntas en español durante una misma ejecución. Carga E5 una vez y reutiliza los vectores de los 60 documentos. Verificamos la integración con dos consultas de referencia y registramos la versión del repositorio y los tres primeros resultados de cada búsqueda.

In [ ]:
%%bash
set -e
cd /content/softtek-knowledge-hub

git pull --ff-only origin main
printf 'Versión evaluada: '
git rev-parse --short HEAD

python -m pip --python .venv/bin/python install -q \
  --index-url https://download.pytorch.org/whl/cpu torch
python -m pip --python .venv/bin/python install -q -r requirements-search.txt

.venv/bin/python src/semantic_search.py --multilingual -k 3 \
  "¿Qué debo hacer para pedir días de vacaciones?" \
  "¿Por qué los pedidos se procesaron con tres horas de retraso?"

Updating afe0322..7c2f7d3
Fast-forward
 src/requirements-search.txt => requirements-search.txt | 0
 1 file changed, 0 insertions(+), 0 deletions(-)
 rename src/requirements-search.txt => requirements-search.txt (100%)
Versión evaluada: 7c2f7d3
E5: 60 documentos indexados para esta sesión.

Query: "¿Qué debo hacer para pedir días de vacaciones?"
  1. [0.837] PP08 - Vacation and Leave Policy (Policies & Procedures)
  2. [0.784] PP03 - Travel and Expense Reimbursement Procedure (Policies & Procedures)
  3. [0.782] TM01 - New Employee Onboarding Guide (Training Manuals)

Query: "¿Por qué los pedidos se procesaron con tres horas de retraso?"
  1. [0.801] IR07 - Post-mortem: Message Queue Backlog (Incident Reports)
  2. [0.800] IR11 - Incident Report: Third-Party Email Service Outage (Incident Reports)
  3. [0.790] IR01 - Post-mortem: Payments API Outage (Incident Reports)


From https://github.com/ierisaac2005-png/softtek-knowledge-hub
 * branch            fase-2     -> FETCH_HEAD
   afe0322..7c2f7d3  fase-2     -> origin/fase-2
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7934.39it/s]


### Verificación final del prototipo funcional

En el commit `7c2f7d3` de la rama `fase-2`, el buscador E5 indexó los 60 documentos una sola vez y atendió dos consultas consecutivas en español. Para la solicitud de vacaciones recuperó en primer lugar `PP08 — Vacation and Leave Policy` (0.837). Para el retraso de tres horas en los pedidos recuperó `IR07 — Post-mortem: Message Queue Backlog` (0.801).

Con esta ejecución quedaron integradas y probadas en Colab las tres funciones del prototipo: clasificación automática, recomendación de documentos y búsqueda en lenguaje natural.